# landing_eda_compras_gov — EDA Compras.gov (Módulo 09)

> **Exploration only — not part of the production pipeline.**
> Run this to understand the raw data structure before or after Bronze processing.

## Purpose
Understand structure, quality, and anomalies in the Compras.gov raw JSON files
before defining and validating the Bronze schema.

## Central question
*"What is in the Compras.gov raw data, what anomalies exist,
and what design decisions does that force in Bronze?"*

## Engine
DuckDB — same engine used in Bronze pipeline. No pandas.

## Steps
1. Imports and configuration
2. File inventory
3. Schema discovery
4. Key field analysis
5. Nested objects — unidadesRequisitantes
6. Multi-month schema comparison
7. Inconsistencies and design decisions
8. Dynamic summary


## Step 1 — Imports and configuration

In [ ]:
import sys
from pathlib import Path

# --- Resolve root ------------------------------------------------------------
try:
    _nb_file = Path(__vsc_ipynb_file__).resolve()
    _root_candidate = _nb_file.parent.parent.parent  # eda/ → notebooks/ → fornecedor360-local/
except NameError:
    _root_candidate = Path.cwd()
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
        if (candidate / "utils" / "paths.py").exists():
            _root_candidate = candidate
            break

sys.path.insert(0, str(_root_candidate / "utils"))

from paths        import get_project_root, to_sql_path
from duckdb_utils import get_connection

PROJECT_ROOT = get_project_root()
con = get_connection()  # in-memory — EDA never writes

COMPRAS_ROOT  = PROJECT_ROOT / "data" / "raw" / "compras_gov"
# Flat structure: 2021-01.json — no subdirectory
available_months = sorted([f.stem for f in COMPRAS_ROOT.glob("*.json")]) \
                   if COMPRAS_ROOT.exists() else []

SAMPLE_MONTH = available_months[0]  if available_months else None
LATEST_MONTH = available_months[-1] if available_months else None
SAMPLE_PATH  = to_sql_path(COMPRAS_ROOT / f"{SAMPLE_MONTH}.json") if SAMPLE_MONTH else None
LATEST_PATH  = to_sql_path(COMPRAS_ROOT / f"{LATEST_MONTH}.json") if LATEST_MONTH else None

MAX_OBJ = 33554432  # 32 MB — needed for large unidadesRequisitantes

print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"COMPRAS_ROOT    : {COMPRAS_ROOT}")
print(f"Months available: {len(available_months)} ({SAMPLE_MONTH} → {LATEST_MONTH})")

## Step 2 — File inventory

**Question:** *How many months do I have? Are all files present and non-empty?
What is the size trend over time?*


In [ ]:
print("=== File Inventory ===")
print()
print(f"{'month':<12} {'size_mb':>8} {'records':>12}")
print("-" * 36)

total_mb, total_rec = 0.0, 0
missing = []

for ym in available_months:
    f = COMPRAS_ROOT / f"{ym}.json"
    if not f.exists():
        print(f"{ym:<12} [MISSING]")
        missing.append(ym)
        continue
    size_mb = f.stat().st_size / 1e6
    path    = to_sql_path(f)
    records = con.execute(
        f"SELECT COUNT(*) FROM read_json_auto('{path}', maximum_object_size={MAX_OBJ})"
    ).fetchone()[0]
    total_mb  += size_mb
    total_rec += records
    print(f"{ym:<12} {size_mb:>8.1f} {records:>12,}")

print("-" * 36)
print(f"{'TOTAL':<12} {total_mb:>8.1f} {total_rec:>12,}")
if missing:
    print(f"[WARN] Missing: {missing}")


## Step 3 — Schema discovery

**Question:** *What columns exist in the raw JSON? What are their types?
Which fields have high null rates that need handling in Bronze?*


In [ ]:
print(f"=== Schema Discovery — {SAMPLE_MONTH} ===")
print()

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ}) LIMIT 0"
).fetchall()

total = con.execute(
    f"SELECT COUNT(*) FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ})"
).fetchone()[0]

print(f"Records : {total:,}  |  Columns : {len(schema)}")
print()
print(f"{'column':<40} {'type':<15} {'nulls':>8} {'null%':>6}")
print("-" * 75)

for col, dtype, *_ in schema:
    nulls = con.execute(
        f"SELECT COUNT(*) FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ}) "
        f"WHERE \"{col}\" IS NULL"
    ).fetchone()[0]
    pct = nulls / total * 100 if total else 0
    print(f"{col:<40} {dtype[:15]:<15} {nulls:>8,} {pct:>5.1f}%")


## Step 4 — Key field analysis

**Question:** *Is niFornecedor always a CNPJ (14 digits)?
Are there CPFs (PF suppliers), foreign identifiers, or malformed values?
Are valor fields always positive?*


In [ ]:
print("=== Key Field Analysis ===")
print()

print("--- niFornecedor (identifier of the supplier) ---")
con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE niFornecedor IS NULL)                       AS nulls,
        COUNT(DISTINCT niFornecedor)                                       AS distinct_vals,
        COUNT(*) FILTER (WHERE LENGTH(niFornecedor) = 14)                  AS len_14_cnpj,
        COUNT(*) FILTER (WHERE LENGTH(niFornecedor) = 11)                  AS len_11_cpf,
        COUNT(*) FILTER (WHERE LENGTH(niFornecedor) NOT IN (11,14))        AS other_len,
        COUNT(*) FILTER (WHERE regexp_matches(niFornecedor, '[A-Za-z]'))   AS alphanumeric
    FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ})
""").df()

print()
print("--- valor fields (are there negatives?) ---")
con.execute(f"""
    SELECT
        'valorGlobal'   AS col,
        COUNT(*) FILTER (WHERE valorGlobal IS NULL) AS nulls,
        MIN(TRY_CAST(valorGlobal  AS DOUBLE)) AS min_val,
        MAX(TRY_CAST(valorGlobal  AS DOUBLE)) AS max_val,
        COUNT(*) FILTER (WHERE TRY_CAST(valorGlobal AS DOUBLE) < 0) AS negative
    FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ})
    UNION ALL
    SELECT 'valorParcela',
        COUNT(*) FILTER (WHERE valorParcela IS NULL),
        MIN(TRY_CAST(valorParcela AS DOUBLE)), MAX(TRY_CAST(valorParcela AS DOUBLE)),
        COUNT(*) FILTER (WHERE TRY_CAST(valorParcela AS DOUBLE) < 0)
    FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ})
""").df()

print()
print("--- contratoExcluido distribution ---")
con.execute(f"""
    SELECT contratoExcluido, COUNT(*) AS cnt
    FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ})
    GROUP BY contratoExcluido ORDER BY cnt DESC
""").df()


## Step 5 — Nested objects — unidadesRequisitantes

**Question:** *How does DuckDB infer the type of unidadesRequisitantes?
Is it a STRUCT, LIST, or VARCHAR? How do we safely store it in Bronze
without losing data or causing type errors across months?*


In [ ]:
import json as json_lib

print("=== unidadesRequisitantes ===")
print()

ur_type = {
    row[0]: row[1]
    for row in con.execute(
        f"DESCRIBE SELECT unidadesRequisitantes "
        f"FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ}) LIMIT 0"
    ).fetchall()
}.get("unidadesRequisitantes", "NOT FOUND")
print(f"DuckDB inferred type : {ur_type}")
print()

con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE unidadesRequisitantes IS NULL)     AS null_count,
        COUNT(*) FILTER (WHERE unidadesRequisitantes IS NOT NULL) AS non_null_count
    FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ})
""").df()

print()
print("Sample non-null values (raw):")
con.execute(f"""
    SELECT CAST(unidadesRequisitantes AS VARCHAR) AS ur_raw
    FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ})
    WHERE unidadesRequisitantes IS NOT NULL
    LIMIT 3
""").df()


## Step 6 — Multi-month schema comparison

**Question:** *Has the schema changed between the oldest and most recent month?
Are there new columns in recent months or columns that disappeared?
Is the Bronze schema stable enough for a wildcard read?*


In [ ]:
print(f"=== Multi-month Schema Comparison ===")
print(f"Oldest: {SAMPLE_MONTH}  |  Most recent: {LATEST_MONTH}")
print()

old_schema = {row[0]: row[1] for row in con.execute(
    f"DESCRIBE SELECT * FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ}) LIMIT 0"
).fetchall()}

new_schema = {row[0]: row[1] for row in con.execute(
    f"DESCRIBE SELECT * FROM read_json_auto('{LATEST_PATH}', maximum_object_size={MAX_OBJ}) LIMIT 0"
).fetchall()}

only_old = sorted(set(old_schema) - set(new_schema))
only_new = sorted(set(new_schema) - set(old_schema))
shared   = sorted(set(old_schema) & set(new_schema))

type_mismatch = [(c, old_schema[c], new_schema[c]) for c in shared if old_schema[c] != new_schema[c]]

print(f"Columns in {SAMPLE_MONTH} only : {only_old or 'none'}")
print(f"Columns in {LATEST_MONTH} only  : {only_new or 'none'}")
print(f"Shared columns               : {len(shared)}")
print(f"Type mismatches              : {type_mismatch or 'none'}")


## Step 7 — Inconsistencies and design decisions

**Question:** *What anomalies require explicit handling in Bronze?
What were the architectural decisions made and why?*


In [ ]:
schema_dict = {row[0]: row[1] for row in con.execute(
    f"DESCRIBE SELECT * FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ}) LIMIT 0"
).fetchall()}

print("=== Inconsistencies and Design Decisions ===")
print()

print("1. niFornecedor — mixed lengths (CNPJ/CPF/foreign)")
print("   Bronze: STRING — alphanumeric CNPJ incoming July 2026 (ADR-002)")
print("   Silver: filter tipoPessoa = PJ for supplier dimension")
print()

print("2. codigoOrgao — returned as STRING despite integer in spec (ADR-006)")
print(f"   DuckDB inferred type: {schema_dict.get('codigoOrgao', 'NOT FOUND')}")
print("   Bronze: force STRING regardless of inference")
print()

print("3. unidadesRequisitantes — dynamic type across months")
print(f"   DuckDB type in sample: {schema_dict.get('unidadesRequisitantes', 'NOT FOUND')}")
print("   Bronze: CAST(to_json(unidadesRequisitantes) AS VARCHAR)")
print("   Silver: json_extract before explode")
print()

print("4. contratoExcluido — keep excluded contracts in Bronze")
print("   Bronze: keep ALL records. Silver: filter WHERE contratoExcluido = false")
print()

print("5. Negative valor fields")
con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE TRY_CAST(valorGlobal  AS DOUBLE) < 0) AS neg_global,
        COUNT(*) FILTER (WHERE TRY_CAST(valorParcela AS DOUBLE) < 0) AS neg_parcela
    FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ})
""").df()
print("   Bronze: keep negatives, add _valor_negativo flag. Silver: filter if needed")
print()

print("6. numeroControlePncpContrato — null in older months (field introduced post-2021)")
cnt = con.execute(f"""
    SELECT COUNT(*) FILTER (WHERE numeroControlePncpContrato IS NULL) AS nulls, COUNT(*) AS total
    FROM read_json_auto('{SAMPLE_PATH}', maximum_object_size={MAX_OBJ})
""").fetchone()
print(f"   Nulls in {SAMPLE_MONTH}: {cnt[0]:,} / {cnt[1]:,} — expected, not schema drift")


## Step 8 — Dynamic summary

**Question:** *What have I learned about this source?
What are the key decisions that downstream notebooks depend on?*


In [ ]:
from IPython.display import display, Markdown

total_files   = len(available_months)
total_records = con.execute(f"""
    SELECT SUM(cnt) FROM (
        {' UNION ALL '.join(
            f"SELECT COUNT(*) AS cnt FROM read_json_auto('{to_sql_path(COMPRAS_ROOT / (m + ".json"))}', maximum_object_size={MAX_OBJ})"
            for m in available_months[:3]
        )}
    )
""").fetchone()[0] if available_months else 0

summary = f"""
## Compras.gov — EDA Summary

| | |
|---|---|
| Files available | {total_files} months ({SAMPLE_MONTH} → {LATEST_MONTH}) |
| Sample records (first 3 months) | {total_records:,} |
| Columns (stable) | {len(schema_dict)} |
| Schema stable across months | {'Yes' if not only_old and not only_new else 'No — see Step 6'} |

### Key Bronze decisions
| Decision | Reason |
|---|---|
| All columns as STRING | Type stability not guaranteed across months |
| `unidadesRequisitantes` → `to_json()` → STRING | Type varies across months |
| `codigoOrgao` as STRING | API returns string despite integer spec (ADR-006) |
| Keep `contratoExcluido = true` | Bronze never filters — Silver decides |
| Add `_valor_negativo` flag | Negative values exist and are meaningful |
| Partition by `_year_month` | One file per month — essential for 65-month dataset |
"""

display(Markdown(summary))
